# Drone Videolarında İnsan Tespiti - Active Learning Frame Selection

Bu notebook eğitim videosu havuzundan **elle etiketlemeye en değerli kareleri** seçer.

Amaç raw DINO çıktısını eğitim etiketi yapmak değil. `dino_tiled` burada öğretmen/önerici olarak kullanılır: hangi karelerde küçük insan var, YOLO nerede kaçırıyor, hangi sahneler belirsiz veya zor gibi sinyallerden bir skor çıkarılır. Seçilen kareler CVAT'a DINO ön-etiketiyle yüklenir, sonra insan tarafından düzeltilir.

Akış:

1. `/kaggle/input` altındaki videoları bulur, test videolarını dışarıda bırakır.
2. Train videolarından seyrek aday kareler örnekler.
3. Her aday karede `DINO-tiled` ve YOLO tahmini çalıştırır.
4. DINO-YOLO uyuşmazlığı, küçük kutu sayısı, DINO belirsizliği ve sahne yoğunluğuna göre active learning skoru hesaplar.
5. Zamansal çeşitlilik filtresiyle birbirinin kopyası kareleri azaltır.
6. Seçilen kareleri ve DINO ön-etiketlerini CVAT formatında dışa aktarır.

Çıktılar `/kaggle/working/active_learning_round1` altında oluşur:

- `images/` - CVAT'a yüklenecek seçilmiş kareler
- `active_learning_images.zip` - seçilmiş görseller
- `active_learning_preannot_cvat.zip` - DINO-tiled ön-etiketleri
- `selection_report.json` ve `selected_frames.csv` - neden seçildiğini açıklayan skorlar

Not: Test setindeki `DJI_0596.MP4`, `Stockflue Flyaround.mp4`, `Surenen Pass Trail Running.mp4` bu notebook'ta kullanılmaz.

In [ ]:
import importlib
import os
import subprocess
import sys

REQUIRED = ["torch", "torchvision", "transformers", "ultralytics", "cv2", "numpy", "PIL"]
for module in REQUIRED:
    try:
        importlib.import_module(module)
    except ImportError:
        if module == "cv2":
            package = "opencv-python-headless"
        elif module == "PIL":
            package = "pillow"
        else:
            package = module
        print(f"Installing {package}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import cv2
import numpy as np
import torch
import transformers
from ultralytics import YOLO

print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemExit("GPU bulunamadi. Settings > Accelerator > GPU T4 secin.")
props = torch.cuda.get_device_properties(0)
print(f"GPU          : {props.name} ({props.total_memory / 1e9:.1f} GB)")

In [ ]:
import inspect
import json
import math
import shutil
import time
import zipfile
from collections import defaultdict
from datetime import datetime, timezone
from xml.etree import ElementTree as ET

from PIL import Image
from torchvision.ops import nms
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

DEFAULT_DINO_MODEL = "IDEA-Research/grounding-dino-base"
PROCESSOR_SHORTEST_EDGE = 800
PROCESSOR_LONGEST_EDGE = 1333
VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv")
CVAT_LABEL = "person"
CVAT_LABEL_COLOR = "#33ddff"


def auto_grid(width, height):
    long_side = max(width, height)
    if long_side >= 3000:
        return 3, 3
    if long_side >= 1200:
        return 2, 2
    return 1, 1


def effective_scale(width, height):
    short_side, long_side = min(width, height), max(width, height)
    scale = PROCESSOR_SHORTEST_EDGE / short_side
    if long_side * scale > PROCESSOR_LONGEST_EDGE:
        scale = PROCESSOR_LONGEST_EDGE / long_side
    return scale


def tile_windows(width, height, rows, cols, overlap=0.2):
    if rows == 1 and cols == 1:
        return [(0, 0, width, height)]
    tile_w = min(width, int(round(width / cols * (1 + overlap))))
    tile_h = min(height, int(round(height / rows * (1 + overlap))))
    step_x = (width - tile_w) / (cols - 1) if cols > 1 else 0
    step_y = (height - tile_h) / (rows - 1) if rows > 1 else 0
    windows = []
    for r in range(rows):
        for c in range(cols):
            x1 = int(round(c * step_x))
            y1 = int(round(r * step_y))
            windows.append((x1, y1, x1 + tile_w, y1 + tile_h))
    return windows


def geometric_filter(boxes, scores, max_area_px, min_area_px=16.0, max_aspect=4.0):
    if len(boxes) == 0:
        return boxes, scores
    widths = boxes[:, 2] - boxes[:, 0]
    heights = boxes[:, 3] - boxes[:, 1]
    areas = widths * heights
    keep = (
        (areas <= max_area_px)
        & (areas >= min_area_px)
        & (widths > 0)
        & (heights > 0)
        & (widths <= heights * max_aspect)
    )
    return boxes[keep], scores[keep]


def suppress_contained(boxes, scores, containment_thresh=0.80):
    if len(boxes) == 0:
        return boxes, scores
    order = np.argsort(-scores)
    boxes, scores = boxes[order], scores[order]
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    keep = np.ones(len(boxes), dtype=bool)
    for i in range(len(boxes)):
        if not keep[i]:
            continue
        for j in range(i + 1, len(boxes)):
            if not keep[j] or areas[j] <= 0:
                continue
            ix1 = max(boxes[i, 0], boxes[j, 0])
            iy1 = max(boxes[i, 1], boxes[j, 1])
            ix2 = min(boxes[i, 2], boxes[j, 2])
            iy2 = min(boxes[i, 3], boxes[j, 3])
            inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
            if inter / areas[j] >= containment_thresh:
                keep[j] = False
    return boxes[keep], scores[keep]


class TiledGroundingDino:
    def __init__(self, model_name=DEFAULT_DINO_MODEL, device=None, use_fp16=False, text_prompt="person."):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype = torch.float16 if (use_fp16 and self.device == "cuda") else torch.float32
        self.text_prompt = text_prompt
        print(f"{model_name} yükleniyor... cihaz={self.device}, dtype={self.dtype}")
        self.processor = AutoProcessor.from_pretrained(model_name)
        try:
            self.model = AutoModelForZeroShotObjectDetection.from_pretrained(model_name, dtype=self.dtype)
        except TypeError:
            self.model = AutoModelForZeroShotObjectDetection.from_pretrained(model_name, torch_dtype=self.dtype)
        self.model = self.model.to(self.device).eval()
        params = inspect.signature(self.processor.post_process_grounded_object_detection).parameters
        self._box_thresh_kwarg = "threshold" if "threshold" in params else "box_threshold"

    def _forward(self, crops_bgr, threshold, text_threshold):
        pil_images = [Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c in crops_bgr]
        inputs = self.processor(
            images=pil_images,
            text=[self.text_prompt] * len(pil_images),
            return_tensors="pt",
        ).to(self.device)
        if self.dtype == torch.float16:
            inputs["pixel_values"] = inputs["pixel_values"].half()
        with torch.inference_mode():
            outputs = self.model(**inputs)
        results = self.processor.post_process_grounded_object_detection(
            outputs,
            input_ids=inputs["input_ids"],
            text_threshold=text_threshold,
            target_sizes=[(c.shape[0], c.shape[1]) for c in crops_bgr],
            **{self._box_thresh_kwarg: threshold},
        )
        return [
            (r["boxes"].float().cpu().numpy().reshape(-1, 4), r["scores"].float().cpu().numpy().reshape(-1))
            for r in results
        ]

    def detect_tiled(self, frame_bgr, grid=None, overlap=0.2, threshold=0.15,
                     text_threshold=0.15, iou_thresh=0.55, include_full_frame=True,
                     max_area_ratio=0.25, batch_size=4):
        h, w = frame_bgr.shape[:2]
        rows, cols = grid or auto_grid(w, h)
        windows = tile_windows(w, h, rows, cols, overlap)
        tile_w = windows[0][2] - windows[0][0]
        tile_h = windows[0][3] - windows[0][1]
        max_area_px = tile_w * tile_h * max_area_ratio

        crops, offsets = [], []
        for (x1, y1, x2, y2) in windows:
            crops.append(frame_bgr[y1:y2, x1:x2])
            offsets.append((x1, y1))
        if include_full_frame and (rows, cols) != (1, 1):
            crops.append(frame_bgr)
            offsets.append((0, 0))

        all_boxes, all_scores = [], []
        for start in range(0, len(crops), batch_size):
            chunk = crops[start:start + batch_size]
            chunk_offsets = offsets[start:start + batch_size]
            for (boxes, scores), (ox, oy) in zip(self._forward(chunk, threshold, text_threshold), chunk_offsets):
                boxes, scores = geometric_filter(boxes, scores, max_area_px)
                if len(boxes) == 0:
                    continue
                boxes = boxes.copy()
                boxes[:, [0, 2]] += ox
                boxes[:, [1, 3]] += oy
                all_boxes.append(boxes)
                all_scores.append(scores)
        if not all_boxes:
            return np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=np.float32)
        boxes = np.clip(np.concatenate(all_boxes), [0, 0, 0, 0], [w, h, w, h]).astype(np.float32)
        scores = np.concatenate(all_scores).astype(np.float32)
        keep = nms(torch.from_numpy(boxes), torch.from_numpy(scores), iou_thresh).numpy()
        return suppress_contained(boxes[keep], scores[keep])


def box_iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0])
    iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2])
    iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)
    area_a = np.maximum(0, a[:, 2] - a[:, 0]) * np.maximum(0, a[:, 3] - a[:, 1])
    area_b = np.maximum(0, b[:, 2] - b[:, 0]) * np.maximum(0, b[:, 3] - b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / np.maximum(union, 1e-6)


def draw_detections(frame, boxes, scores, color=(0, 255, 0), label_prefix="person"):
    out = frame.copy()
    h, w = out.shape[:2]
    thickness = max(1, round((h + w) / 1600))
    font_scale = max(0.4, thickness * 0.4)
    for (x1, y1, x2, y2), score in zip(boxes, scores):
        p1, p2 = (int(x1), int(y1)), (int(x2), int(y2))
        cv2.rectangle(out, p1, p2, color, thickness)
        label = f"{label_prefix} {score:.2f}"
        cv2.putText(out, label, (p1[0], max(15, p1[1] - 4)), cv2.FONT_HERSHEY_SIMPLEX,
                    font_scale, color, thickness, cv2.LINE_AA)
    return out


def slugify(video_name):
    stem = os.path.splitext(video_name)[0]
    return "".join(ch if ch.isalnum() else "_" for ch in stem).strip("_")


def build_cvat_xml(images, task_name):
    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f+00:00")
    root = ET.Element("annotations")
    ET.SubElement(root, "version").text = "1.1"
    meta = ET.SubElement(root, "meta")
    task = ET.SubElement(meta, "task")
    ET.SubElement(task, "id").text = "1"
    ET.SubElement(task, "name").text = task_name
    ET.SubElement(task, "size").text = str(len(images))
    ET.SubElement(task, "mode").text = "annotation"
    ET.SubElement(task, "overlap").text = "0"
    ET.SubElement(task, "bugtracker")
    ET.SubElement(task, "created").text = now
    ET.SubElement(task, "updated").text = now
    ET.SubElement(task, "start_frame").text = "0"
    ET.SubElement(task, "stop_frame").text = str(max(0, len(images) - 1))
    ET.SubElement(task, "frame_filter")
    labels = ET.SubElement(task, "labels")
    label = ET.SubElement(labels, "label")
    ET.SubElement(label, "name").text = CVAT_LABEL
    ET.SubElement(label, "color").text = CVAT_LABEL_COLOR
    ET.SubElement(label, "type").text = "rectangle"
    ET.SubElement(label, "attributes")
    segments = ET.SubElement(task, "segments")
    segment = ET.SubElement(segments, "segment")
    ET.SubElement(segment, "id").text = "1"
    ET.SubElement(segment, "start").text = "0"
    ET.SubElement(segment, "stop").text = str(max(0, len(images) - 1))
    ET.SubElement(segment, "url")
    owner = ET.SubElement(task, "owner")
    ET.SubElement(owner, "username")
    ET.SubElement(owner, "email")
    ET.SubElement(meta, "dumped").text = now

    for image_id, item in enumerate(images):
        img_el = ET.SubElement(root, "image", {
            "id": str(image_id),
            "name": item["file_name"],
            "width": str(item["width"]),
            "height": str(item["height"]),
        })
        for (x1, y1, x2, y2), score in zip(item["dino_boxes"], item["dino_scores"]):
            ET.SubElement(img_el, "box", {
                "label": CVAT_LABEL,
                "source": "auto",
                "occluded": "0",
                "xtl": f"{x1:.2f}",
                "ytl": f"{y1:.2f}",
                "xbr": f"{x2:.2f}",
                "ybr": f"{y2:.2f}",
                "z_order": "0",
                "score": f"{score:.4f}",
            })
    ET.indent(root, space="  ")
    return ET.ElementTree(root)


def find_videos(root):
    found = {}
    for dirpath, _, filenames in os.walk(root):
        for name in filenames:
            if name.lower().endswith(VIDEO_EXTS):
                found.setdefault(name, os.path.join(dirpath, name))
    return found


def find_yolo_weight(root="/kaggle/input", preferred=("yolo11x.pt", "yolo11n.pt", "yolov8x.pt")):
    for dirpath, _, filenames in os.walk(root):
        for p in preferred:
            if p in filenames:
                return os.path.join(dirpath, p)
    return preferred[0]  # ultralytics internet aciksa indirir

In [ ]:
# Ayarlar
KAGGLE_INPUT = "/kaggle/input"
OUT_ROOT = "/kaggle/working/active_learning_round1"
POOL_DIR = os.path.join(OUT_ROOT, "pool")
SELECTED_DIR = os.path.join(OUT_ROOT, "images")
PREVIEW_DIR = os.path.join(OUT_ROOT, "preview")
ANN_DIR = os.path.join(OUT_ROOT, "annotations")

TEST_VIDEOS = {"DJI_0596.MP4", "Stockflue Flyaround.mp4", "Surenen Pass Trail Running.mp4"}

CANDIDATE_INTERVAL = 30   # 1 fps aday havuzu. Zaman varsa 15 yapabilirsin.
SELECT_COUNT = 120        # Ilk active learning turunda elle etiketlenecek kare sayisi.
MIN_GAP_FRAMES = 60       # Ayni videodan cok yakin kareleri secme (yaklasik 2 saniye).
DINO_THRESHOLD = 0.15     # CVAT on-etiketleri recall oncelikli olsun.
YOLO_CONF = 0.01          # Uyuşmazlık hesabı için düşük eşik.
YOLO_IMGSZ = 1536         # Baseline ölçümde DINO full'u geçen güçlü YOLO ayarı.
TILE_BATCH_SIZE = 4       # T4 için. OOM alırsan 2'ye düşür.
TEXT_PROMPT = "person."

for d in (POOL_DIR, SELECTED_DIR, PREVIEW_DIR, ANN_DIR):
    os.makedirs(d, exist_ok=True)

available = find_videos(KAGGLE_INPUT)
if not available:
    raise SystemExit("/kaggle/input altinda video yok. Add Input > Datasets ile dataset ekleyin.")

train_videos = sorted(set(available) - TEST_VIDEOS)
print(f"Bulunan video sayisi: {len(available)}")
print(f"Test disi egitim havuzu: {len(train_videos)} video")
for name in train_videos:
    print("  ", name)
print("Test videolari disarida birakildi:", sorted(TEST_VIDEOS))

# Aday kareleri diske yaz.
frame_records = []
print("\nAday kareler ornekleniyor...")
for video_name in train_videos:
    path = available[video_name]
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        print(f"UYARI: acilamadi: {video_name}")
        continue
    slug = slugify(video_name)
    frame_idx = 0
    saved = 0
    first_width, first_height = None, None
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_idx % CANDIDATE_INTERVAL == 0:
            file_name = f"{slug}_f{frame_idx:06d}.jpg"
            out_path = os.path.join(POOL_DIR, file_name)
            cv2.imwrite(out_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
            first_width = first_width or frame.shape[1]
            first_height = first_height or frame.shape[0]
            frame_records.append({
                "file_name": file_name,
                "video": video_name,
                "frame_index": frame_idx,
                "width": frame.shape[1],
                "height": frame.shape[0],
                "pool_path": out_path,
            })
            saved += 1
        frame_idx += 1
    cap.release()
    rows, cols = auto_grid(first_width, first_height) if saved else (0, 0)
    print(f"  {video_name:<34} -> {saved:>3} kare | dosem {rows}x{cols}")

print(f"\nToplam aday kare: {len(frame_records)}")

# Modelleri yükle.
yolo_weight = find_yolo_weight()
print(f"\nYOLO agirligi: {yolo_weight}")
yolo = YOLO(yolo_weight)
teacher = TiledGroundingDino(use_fp16=False, text_prompt=TEXT_PROMPT)

# Her aday kare için DINO-tiled ve YOLO tahminlerini çıkar, active learning skorunu hesapla.
start = time.time()
for i, record in enumerate(frame_records, 1):
    frame = cv2.imread(record["pool_path"])
    h, w = frame.shape[:2]

    dino_boxes, dino_scores = teacher.detect_tiled(
        frame,
        threshold=DINO_THRESHOLD,
        text_threshold=DINO_THRESHOLD,
        batch_size=TILE_BATCH_SIZE,
    )
    yolo_result = yolo.predict(frame, imgsz=YOLO_IMGSZ, classes=[0], conf=YOLO_CONF, verbose=False)[0]
    if len(yolo_result.boxes):
        yolo_boxes = yolo_result.boxes.xyxy.cpu().numpy().astype(np.float32)
        yolo_scores = yolo_result.boxes.conf.cpu().numpy().astype(np.float32)
    else:
        yolo_boxes = np.zeros((0, 4), dtype=np.float32)
        yolo_scores = np.zeros((0,), dtype=np.float32)

    ious = box_iou_matrix(dino_boxes, yolo_boxes)
    dino_matched = (ious.max(axis=1) >= 0.30) if len(dino_boxes) and len(yolo_boxes) else np.zeros(len(dino_boxes), dtype=bool)
    yolo_matched = (ious.max(axis=0) >= 0.30) if len(dino_boxes) and len(yolo_boxes) else np.zeros(len(yolo_boxes), dtype=bool)

    dino_areas = (dino_boxes[:, 2] - dino_boxes[:, 0]) * (dino_boxes[:, 3] - dino_boxes[:, 1]) if len(dino_boxes) else np.array([])
    small_dino = int((dino_areas < 32 * 32).sum())
    medium_uncertain = int(((dino_scores >= 0.15) & (dino_scores < 0.35)).sum())
    missed_by_yolo = int((~dino_matched).sum())
    yolo_only = int((~yolo_matched).sum())
    crowded = int(len(dino_boxes) >= 8)

    # Skor yorumu:
    # - missed_by_yolo: YOLO'nun ogrenmesi gereken DINO destekli potansiyel insan
    # - small_dino: AP_small'i dogrudan hedefler
    # - medium_uncertain: insan etiketi en cok bilgiyi burada verir
    # - yolo_only: YOLO'nun false positive uretebilecegi zor arka planlari yakalar
    # - crowded: kalabalik sahnelerde hem kacirma hem duplicate hatasi artar
    al_score = (
        4.0 * missed_by_yolo
        + 2.5 * small_dino
        + 1.5 * medium_uncertain
        + 1.0 * yolo_only
        + 2.0 * crowded
        + 0.2 * len(dino_boxes)
    )

    record.update({
        "dino_boxes": dino_boxes.tolist(),
        "dino_scores": dino_scores.tolist(),
        "yolo_boxes": yolo_boxes.tolist(),
        "yolo_scores": yolo_scores.tolist(),
        "al_score": float(al_score),
        "dino_count": int(len(dino_boxes)),
        "yolo_count": int(len(yolo_boxes)),
        "missed_by_yolo": missed_by_yolo,
        "yolo_only": yolo_only,
        "small_dino": small_dino,
        "medium_uncertain": medium_uncertain,
        "crowded": crowded,
    })

    if i % 10 == 0 or i == len(frame_records):
        elapsed = time.time() - start
        print(f"  {i}/{len(frame_records)} | {elapsed / i:.2f} s/kare | kalan ~{(len(frame_records)-i)*elapsed/i/60:.1f} dk")

In [ ]:
# En yüksek skorlu kareleri seç, ama aynı videodan birbirine çok yakın kareleri azalt.
def select_diverse(records, budget, min_gap_frames):
    selected = []
    per_video_frames = defaultdict(list)
    sorted_records = sorted(records, key=lambda r: -r["al_score"])

    for gap in (min_gap_frames, min_gap_frames // 2, 0):
        for r in sorted_records:
            if len(selected) >= budget:
                break
            if r.get("selected"):
                continue
            previous = per_video_frames[r["video"]]
            if gap and any(abs(r["frame_index"] - p) < gap for p in previous):
                continue
            r["selected"] = True
            selected.append(r)
            previous.append(r["frame_index"])
        if len(selected) >= budget:
            break
    for r in records:
        r.pop("selected", None)
    return sorted(selected, key=lambda r: (r["video"], r["frame_index"]))


selected = select_diverse(frame_records, SELECT_COUNT, MIN_GAP_FRAMES)
print(f"Secilen kare: {len(selected)} / {len(frame_records)}")

# Seçilen görselleri ayrı klasöre kopyala; CVAT sadece bu klasörü görecek.
for r in selected:
    dst = os.path.join(SELECTED_DIR, r["file_name"])
    shutil.copy2(r["pool_path"], dst)

# CVAT XML ve zip.
xml_path = os.path.join(ANN_DIR, "active_learning_preannot_cvat.xml")
build_cvat_xml(selected, "drone_person_active_learning_round1").write(
    xml_path, encoding="utf-8", xml_declaration=True
)
cvat_zip = os.path.join(ANN_DIR, "active_learning_preannot_cvat.zip")
with zipfile.ZipFile(cvat_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(xml_path, "annotations.xml")

images_zip = os.path.join(OUT_ROOT, "active_learning_images.zip")
with zipfile.ZipFile(images_zip, "w", zipfile.ZIP_STORED) as zf:
    for r in selected:
        zf.write(os.path.join(SELECTED_DIR, r["file_name"]), r["file_name"])

# CSV + JSON rapor.
csv_path = os.path.join(OUT_ROOT, "selected_frames.csv")
columns = [
    "file_name", "video", "frame_index", "al_score", "dino_count", "yolo_count",
    "missed_by_yolo", "small_dino", "medium_uncertain", "yolo_only", "crowded",
]
with open(csv_path, "w", encoding="utf-8") as f:
    f.write(",".join(columns) + "\n")
    for r in selected:
        f.write(",".join(str(round(r[c], 4)) if isinstance(r[c], float) else str(r[c]) for c in columns) + "\n")

report_path = os.path.join(OUT_ROOT, "selection_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump({
        "purpose": "Active learning round 1 frame selection",
        "test_videos_excluded": sorted(TEST_VIDEOS),
        "config": {
            "candidate_interval": CANDIDATE_INTERVAL,
            "select_count": SELECT_COUNT,
            "min_gap_frames": MIN_GAP_FRAMES,
            "dino_threshold": DINO_THRESHOLD,
            "yolo_conf": YOLO_CONF,
            "yolo_imgsz": YOLO_IMGSZ,
            "tile_batch_size": TILE_BATCH_SIZE,
            "yolo_weight": yolo_weight,
        },
        "all_candidates": frame_records,
        "selected": selected,
    }, f, indent=2)

# Kısa özet.
per_video = defaultdict(lambda: {"selected": 0, "score_sum": 0.0, "dino": 0, "missed": 0, "small": 0})
for r in selected:
    s = per_video[r["video"]]
    s["selected"] += 1
    s["score_sum"] += r["al_score"]
    s["dino"] += r["dino_count"]
    s["missed"] += r["missed_by_yolo"]
    s["small"] += r["small_dino"]

print("\nVideo bazli secim ozeti")
print(f"{'Video':<34} | {'Secilen':>7} | {'Ort skor':>8} | {'DINO':>5} | {'YOLO kacirdi':>12} | {'Small':>5}")
print("-" * 88)
for video, s in sorted(per_video.items()):
    print(f"{video:<34} | {s['selected']:>7} | {s['score_sum']/s['selected']:>8.1f} | "
          f"{s['dino']:>5} | {s['missed']:>12} | {s['small']:>5}")

print("\nCiktilar:")
for p in (images_zip, cvat_zip, csv_path, report_path):
    print(f"  {os.path.getsize(p)/1e6:8.1f} MB  {p}")

In [ ]:
import matplotlib.pyplot as plt

# En yüksek skorlu birkaç kareyi DINO ön-etiketleriyle çiz.
preview_items = sorted(selected, key=lambda r: -r["al_score"])[:8]
for r in preview_items:
    frame = cv2.imread(os.path.join(SELECTED_DIR, r["file_name"]))
    annotated = draw_detections(
        frame,
        np.asarray(r["dino_boxes"], dtype=np.float32).reshape(-1, 4),
        np.asarray(r["dino_scores"], dtype=np.float32).reshape(-1),
    )
    scale = 1600 / max(annotated.shape[:2])
    if scale < 1:
        annotated = cv2.resize(annotated, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    cv2.imwrite(os.path.join(PREVIEW_DIR, r["file_name"]), annotated)

cols = 2
rows = math.ceil(len(preview_items) / cols)
if preview_items:
    fig, axes = plt.subplots(rows, cols, figsize=(18, 7 * rows))
    axes = np.asarray(axes).reshape(-1)
    for ax, r in zip(axes, preview_items):
        img = cv2.imread(os.path.join(PREVIEW_DIR, r["file_name"]))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(
            f"{r['file_name']} | score={r['al_score']:.1f} | "
            f"DINO={r['dino_count']} YOLO={r['yolo_count']} missed={r['missed_by_yolo']}",
            fontsize=10,
        )
        ax.axis("off")
    for ax in axes[len(preview_items):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Secilen kare yok.")

print(f"Onizleme klasoru: {PREVIEW_DIR}")